# Option 3 – Inverted Index from Wikipedia Pages

**Assignment:**
- A.1. Choose 5 topics, each with 3 Wikipedia pages (15 pages total)
- A.2. Extract 10 keywords from each page
- B. Build an inverted index mapping each keyword → list of URLs where it appears

**Topics chosen:**
1. Artificial Intelligence
2. Space Exploration
3. Climate Change
4. Ancient Rome
5. Music Theory

## Step 0 – Imports and Setup

In [1]:
import requests
import time
import re
from collections import defaultdict
import pandas as pd

# Stop-words to ignore when extracting meaningful keywords
STOP_WORDS = {
    'the','a','an','is','it','in','on','of','to','and','or','for',
    'with','as','at','by','from','that','this','are','was','were',
    'be','been','has','have','had','its','which','also','not','but',
    'he','she','they','we','his','her','their','more','than','can',
    'other','some','such','when','into','used','about','may','these',
    'between','known','well','often','while','both','however','would'
}

print('Imports loaded successfully.')

Imports loaded successfully.


## Step 1 – Define the 5 Topics and their Wikipedia URLs

Each topic has 3 Wikipedia pages (15 pages total).

In [8]:
topics = {
    'Artificial Intelligence': [
        'https://en.wikipedia.org/wiki/Artificial_intelligence',
        'https://en.wikipedia.org/wiki/Machine_learning',
        'https://en.wikipedia.org/wiki/Neural_network_(machine_learning)',
    ],
    'Space Exploration': [
        'https://en.wikipedia.org/wiki/Space_exploration',
        'https://en.wikipedia.org/wiki/Apollo_program',
        'https://en.wikipedia.org/wiki/International_Space_Station',
    ],
    'Climate Change': [
        'https://en.wikipedia.org/wiki/Climate_change',
        'https://en.wikipedia.org/wiki/Greenhouse_gas',
        'https://en.wikipedia.org/wiki/Effects_of_climate_change',
    ],
    'Ancient Rome': [
        'https://en.wikipedia.org/wiki/Ancient_Rome',
        'https://en.wikipedia.org/wiki/Roman_Empire',
        'https://en.wikipedia.org/wiki/Julius_Caesar',
    ],
    'Music Theory': [
        'https://en.wikipedia.org/wiki/Music_theory',
        'https://en.wikipedia.org/wiki/Harmony',
        'https://en.wikipedia.org/wiki/Scale_(music)',
    ],
}

all_urls = [url for urls in topics.values() for url in urls]

for topic, urls in topics.items():
    print(f'Topic: {topic}')
    for url in urls:
        print(f'  - {url}')
    print()

print(f'Total pages: {len(all_urls)}')

Topic: Artificial Intelligence
  - https://en.wikipedia.org/wiki/Artificial_intelligence
  - https://en.wikipedia.org/wiki/Machine_learning
  - https://en.wikipedia.org/wiki/Neural_network_(machine_learning)

Topic: Space Exploration
  - https://en.wikipedia.org/wiki/Space_exploration
  - https://en.wikipedia.org/wiki/Apollo_program
  - https://en.wikipedia.org/wiki/International_Space_Station

Topic: Climate Change
  - https://en.wikipedia.org/wiki/Climate_change
  - https://en.wikipedia.org/wiki/Greenhouse_gas
  - https://en.wikipedia.org/wiki/Effects_of_climate_change

Topic: Ancient Rome
  - https://en.wikipedia.org/wiki/Ancient_Rome
  - https://en.wikipedia.org/wiki/Roman_Empire
  - https://en.wikipedia.org/wiki/Julius_Caesar

Topic: Music Theory
  - https://en.wikipedia.org/wiki/Music_theory
  - https://en.wikipedia.org/wiki/Harmony
  - https://en.wikipedia.org/wiki/Scale_(music)

Total pages: 15


## Step 2 – Fetch Page Text via Wikipedia API

We use the Wikipedia REST API (`/w/api.php`) to get plain text for each page.  
We wait **5–10 seconds** between each request to be polite to the server.

In [9]:
def get_wiki_text(page_title):
    """Fetch plain text of a Wikipedia article using the MediaWiki API."""
    api_url = 'https://en.wikipedia.org/w/api.php'
    params = {
        'action': 'query',
        'titles': page_title,
        'prop': 'extracts',
        'explaintext': True,
        'format': 'json'
    }
    headers = {'User-Agent': 'InvertedIndexAssignment/1.0 (student project)'}

    for attempt in range(3):
        response = requests.get(api_url, params=params, headers=headers)
        data = response.json()
        page = next(iter(data['query']['pages'].values()))

        if 'missing' in page:
            print(f'  WARNING: page not found - {page_title}')
            return ''

        text = page.get('extract', '')
        if text:
            return text

        print(f'  Empty result, retrying in 10s... (attempt {attempt+1})')
        time.sleep(10)

    return ''


page_texts = {}  # url -> text

for i, url in enumerate(all_urls):
    # Extract the page title from the URL
    title = url.split('/wiki/')[-1]
    print(f'Fetching: {title} ...', end=' ')

    text = get_wiki_text(title)
    page_texts[url] = text
    print(f'OK (length: {len(text)})')

    # Wait 5 seconds between requests (except after the last one)
    if i < len(all_urls) - 1:
        print('Waiting 5 seconds...')
        time.sleep(5)

print(f'\nDone! Fetched {len(page_texts)} pages.')

Fetching: Artificial_intelligence ... OK (length: 84284)
Waiting 5 seconds...
Fetching: Machine_learning ... OK (length: 59423)
Waiting 5 seconds...
Fetching: Neural_network_(machine_learning) ... OK (length: 62899)
Waiting 5 seconds...
Fetching: Space_exploration ... OK (length: 43669)
Waiting 5 seconds...
Fetching: Apollo_program ... OK (length: 65510)
Waiting 5 seconds...
Fetching: International_Space_Station ... OK (length: 117701)
Waiting 5 seconds...
Fetching: Climate_change ... OK (length: 64743)
Waiting 5 seconds...
Fetching: Greenhouse_gas ... OK (length: 19308)
Waiting 5 seconds...
Fetching: Effects_of_climate_change ... OK (length: 50631)
Waiting 5 seconds...
Fetching: Ancient_Rome ... OK (length: 97083)
Waiting 5 seconds...
Fetching: Roman_Empire ... OK (length: 101205)
Waiting 5 seconds...
Fetching: Julius_Caesar ... OK (length: 64671)
Waiting 5 seconds...
Fetching: Music_theory ... OK (length: 56163)
Waiting 5 seconds...
Fetching: Harmony ... OK (length: 25241)
Waiting 5 

## Step 3 – Extract Top 10 Keywords per Page

We tokenize the text, remove stop-words, and pick the 10 most frequent words (at least 5 characters long) as keywords for each page.

In [10]:
def extract_keywords(text, top_n=10, min_len=5):
    """Return the top_n most frequent non-stopword words of length >= min_len."""
    words = re.findall(r'[a-zA-Z]+', text.lower())
    freq = defaultdict(int)
    for w in words:
        if w not in STOP_WORDS and len(w) >= min_len:
            freq[w] += 1
    sorted_words = sorted(freq, key=freq.get, reverse=True)
    return sorted_words[:top_n]


page_keywords = {}  # url -> list of 10 keywords

for url, text in page_texts.items():
    keywords = extract_keywords(text)
    page_keywords[url] = keywords

print('Keywords per page:\n')
for url, kw in page_keywords.items():
    print(url)
    print(f'  {kw}\n')

Keywords per page:

https://en.wikipedia.org/wiki/Artificial_intelligence
  ['intelligence', 'learning', 'artificial', 'human', 'machine', 'problem', 'research', 'search', 'networks', 'problems']

https://en.wikipedia.org/wiki/Machine_learning
  ['learning', 'machine', 'algorithms', 'model', 'training', 'artificial', 'models', 'systems', 'methods', 'algorithm']

https://en.wikipedia.org/wiki/Neural_network_(machine_learning)
  ['neural', 'learning', 'networks', 'network', 'model', 'artificial', 'training', 'neurons', 'function', 'layer']

https://en.wikipedia.org/wiki/Space_exploration
  ['space', 'first', 'exploration', 'earth', 'spacecraft', 'human', 'mission', 'orbit', 'spaceflight', 'missions']

https://en.wikipedia.org/wiki/Apollo_program
  ['apollo', 'lunar', 'program', 'saturn', 'mission', 'first', 'space', 'module', 'earth', 'flight']

https://en.wikipedia.org/wiki/International_Space_Station
  ['station', 'space', 'module', 'russian', 'spacecraft', 'earth', 'system', 'orbital'

## Step 4 – Build the Inverted Index

The inverted index maps each keyword to a list of URLs (pages) where it was found as one of the top 10 keywords.

In [11]:
inverted_index = defaultdict(list)  # keyword -> [url1, url2, ...]

for url, keywords in page_keywords.items():
    for kw in keywords:
        inverted_index[kw].append(url)

# Sort alphabetically for readability
inverted_index = dict(sorted(inverted_index.items()))

print('Inverted index built.')
print(f'Total unique keywords: {len(inverted_index)}')

Inverted index built.
Total unique keywords: 104


## Step 5 – Display the Inverted Index as a DataFrame

In [12]:
rows = []
for kw, urls in inverted_index.items():
    rows.append({
        'Keyword': kw,
        'Appears In (URLs)': ', '.join(urls),
        'Count': len(urls)
    })

df_index = pd.DataFrame(rows)

pd.set_option('display.max_colwidth', None)
display(df_index)

,Keyword,Appears In (URLs),Count
0,after,"https://en.wikipedia.org/wiki/Ancient_Rome, https://en.wikipedia.org/wiki/Julius_Caesar",2
1,algorithm,https://en.wikipedia.org/wiki/Machine_learning,1
2,algorithms,https://en.wikipedia.org/wiki/Machine_learning,1
3,analysis,https://en.wikipedia.org/wiki/Music_theory,1
4,ancient,https://en.wikipedia.org/wiki/Ancient_Rome,1
...,...,...,...
99,tonic,https://en.wikipedia.org/wiki/Scale_(music),1
100,training,"https://en.wikipedia.org/wiki/Machine_learning, https://en.wikipedia.org/wiki/Neural_network_(machine_learning)",2
101,under,https://en.wikipedia.org/wiki/Ancient_Rome,1
102,warming,"https://en.wikipedia.org/wiki/Climate_change, https://en.wikipedia.org/wiki/Effects_of_climate_change",2


## Step 6 – Show Keywords That Appear in Multiple Pages

These keywords are the most interesting because they cross-link pages from the same or different topics.

In [13]:
df_multi = df_index[df_index['Count'] >= 2].sort_values('Count', ascending=False)
df_multi = df_multi.reset_index(drop=True)

print('Keywords appearing in 2+ pages:')
display(df_multi[['Keyword', 'Count', 'Appears In (URLs)']])

Keywords appearing in 2+ pages:


,Keyword,Count,Appears In (URLs)
0,artificial,3,"https://en.wikipedia.org/wiki/Artificial_intelligence, https://en.wikipedia.org/wiki/Machine_learning, https://en.wikipedia.org/wiki/Neural_network_(machine_learning)"
1,first,3,"https://en.wikipedia.org/wiki/Space_exploration, https://en.wikipedia.org/wiki/Apollo_program, https://en.wikipedia.org/wiki/Julius_Caesar"
2,earth,3,"https://en.wikipedia.org/wiki/Space_exploration, https://en.wikipedia.org/wiki/Apollo_program, https://en.wikipedia.org/wiki/International_Space_Station"
3,climate,3,"https://en.wikipedia.org/wiki/Climate_change, https://en.wikipedia.org/wiki/Greenhouse_gas, https://en.wikipedia.org/wiki/Effects_of_climate_change"
4,during,3,"https://en.wikipedia.org/wiki/International_Space_Station, https://en.wikipedia.org/wiki/Ancient_Rome, https://en.wikipedia.org/wiki/Julius_Caesar"
5,learning,3,"https://en.wikipedia.org/wiki/Artificial_intelligence, https://en.wikipedia.org/wiki/Machine_learning, https://en.wikipedia.org/wiki/Neural_network_(machine_learning)"
6,roman,3,"https://en.wikipedia.org/wiki/Ancient_Rome, https://en.wikipedia.org/wiki/Roman_Empire, https://en.wikipedia.org/wiki/Julius_Caesar"
7,space,3,"https://en.wikipedia.org/wiki/Space_exploration, https://en.wikipedia.org/wiki/Apollo_program, https://en.wikipedia.org/wiki/International_Space_Station"
8,music,3,"https://en.wikipedia.org/wiki/Music_theory, https://en.wikipedia.org/wiki/Harmony, https://en.wikipedia.org/wiki/Scale_(music)"
9,change,2,"https://en.wikipedia.org/wiki/Climate_change, https://en.wikipedia.org/wiki/Effects_of_climate_change"
